# DAXM wires calibration: crude from the Laue spots dips, fine with the fluorescence

The wires trajectory parameters (`.calib` file: `h`, `p0`, `f1`, `f2`, `R` of each wire, `u1`, `u2`) are needed to convert the dips of the Laue spots (quick daxm) or the shadowed intensities (reconstruction) into positions of the sources along the beam.

| step | what | result (in `PROCESSED_DATA/<sample>/<dataset>/scanXXXX/daxm`) |
|---|---|---|
| 1 | choose the wire scan used for the calibration (a Ge crystal, or a scan of the sample itself) | |
| 2 | settings file | `configs/daxm.yaml` |
| 3 | quick daxm: dips of the Laue spots (no wires calibration) | `quickdaxm.h5` |
| 4 | **crude calibration (seconds)** from the dips of the Laue spots: `dips` (Ge: all the sources at the surface, fit of the wires to the dips edges) or `envelope` (any sample: the shallowest grains seen by each wire define the depth 0) | `wirescalib/wires_from_dips.calib` or `wires_from_envelope.calib` |
| 5 | **fine calibration (minutes, option)**: refinement with the fluorescence intensity profiles (usual calibration, `calib.py` stages), starting from step 4 | `wirescalib/wires_fluo_stage<n>.calib` |
| 6 | comparison of the calibrations: wires parameters, positions of the Laue spots along the beam | |

The wire scans of the same sample folder use this calibration automatically (`CALIBFILE = 'auto'` in `2_quick_daxm` and `3_quick_daxm_grid`): the refined one if done, else the crude one.

**Depth zero of each calibration**: `dips` (Ge): the mean depth of the Ge Laue spots; `envelope`: the shallowest grains (e.g. top of a coating); fluorescence: the top of the fluorescent element (e.g. Zr below a Cr coating if the Cr fluorescence is cut by the detector energy threshold). Step 6 shows the shift between them.

Core functions: `LaueTools/Daxm/quickdaxm.py`, `LaueTools/daxm_workflow.py`; fluorescence refinement: legacy `CalibFluo` (`LaueTools/Daxm/classes/calibration`).

# SETTINGS

In [ ]:
# 1. experiment and wire scan of the calibration
RAW_DATA_FOLDER = '/data/visitor/<proposal>/bm32/<date>/RAW_DATA'
SCAN_FILTER = '<sample>_grid'  # words of the dataset name ('-word': without it); '': all wire scans
SCAN_INDEX = '<sample>_grid_index_0_row_col_0_0_1'  # None: menu of STEP 1; or index in the table or dataset name
MODE = 'compute'           # 'compute'; 'show': only show existing results
DETFILE = 'auto'           # detector calibration .det file; 'auto': the last one acquired before the scan

# 2. crude calibration from the dips of the Laue spots (STEP 4)
CRUDE_METHOD = 'auto'      # 'dips' (Ge: sources at the surface), 'envelope' (sample: shallowest grains = depth 0),
                           # 'auto': 'dips' for a Ge scan (Ge in the dataset name), else 'envelope'
NB_WIRES = None            # nb of wires (None: all the bands of dips found)
RADIUS_EACH_WIRE = False   # 'dips': radius of each wire fitted (wires of different diameters)
ENVELOPE_OUTLIERS = 0.02   # 'envelope': fraction of the dips of a wire allowed beyond its envelope line
EXCLUDED_WIRES = None      # wires whose dips give no depth (kept in the .calib: their shadows are known), e.g. [3]

# 3. fine calibration with the fluorescence (STEP 5)
CHECK_FLUO_PROFILES = True # plots of the fluorescence profiles used by the refinement (~1 min)
REPLACE_BAD_POINTS = True  # unusable profiles (Laue spot in the box, dip cut by the scan, not a fluorescence dip)
                           # replaced by usable ones probed nearby (along X first), in the check and the refinement
PROBE_ALL_POINTS = False   # True: every point probed (moved only if its profile is clearly better)
FLUO_RESOLUTION = False    # blur of the fluorescence dips with the refined wires (depth resolution, ~1 min)
REFINE_FLUO = False        # refinement (several minutes per stage)
FLUO_ELEMENT = 'Zr'        # fluorescent element of the scanned sample (Ge for a Ge crystal)
FLUO_THICKNESS = 0.3       # mm, thickness of the fluorescent layer (model of the source)
FLUO_LAYER = 0.025         # mm along the beam: layer above the fluorescent material whose signal passes the detector
                           # threshold (Cr coating on Zr: ~0.025), fitted with the wires; None: no layer (Ge)
FLUO_WIRES = None          # wires refined, e.g. [0, 1, 2] (the others keep their crude parameters, excluded); None: all
FLUO_STAGES = 3            # nb of stages of the refinement: 1 (2x2 points per wire), 2 (3x3), 3 (6x6)
FLUO_RESOURCE = 'kernel'   # where the refinement runs: 'kernel' (this notebook), 'local' (background process here),
                           # 'magnifix' (SLURM job, 192 cpus for the jacobian); the notebook waits for the result

# 4. computation and files
QUICKDAXM = {'skip_images': 40}  # changes of the quick daxm parameters. skip_images: first images of the scans not used
                           # (Zr5 grid: wire position wrong by ~8 µm in images 0-39 of the scans after the first one,
                           # see mismatch_vs_image())
NB_CPUS = None             # None: all available cpus
REDO = False               # True: compute again the quick daxm of the scan
OUTPUT_ROOT = None         # None: PROCESSED_DATA of the experiment; or another folder for the results

# IMPORTS

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
try:
    import ipympl  # interactive plots (zoom, hover) in JupyterLab / VS Code
    %matplotlib widget
except ImportError:
    %matplotlib inline
from IPython.display import display

import LaueTools.daxm_workflow as DW
import LaueTools.Daxm.quickdaxm as QD
import LaueTools.generaltools as GT

# STEP 1: wire scan of the calibration

A Ge wire scan (reference crystal), or a wire scan of the sample (its own Laue spots dips, its own fluorescence). `done`: results already there (`wires:dips`, `wires:envelope`, `wires:fluo`: calibrations made with this scan).

In [ ]:
wirescans = DW.list_wire_scans(RAW_DATA_FOLDER, contains=SCAN_FILTER, output_root=OUTPUT_ROOT)
DW.show_wire_scans(wirescans)

In [ ]:
selector = DW.WireScanSelector(wirescans, SCAN_INDEX)

# STEP 2: settings file (without wires calibration)

In [ ]:
index = selector.index
row = wirescans.loc[index]
print(f"scan {index}: {row['dataset']}  {row['command']}  ({row['end']})")
if MODE == 'show':
    cfg = DW.load_config(Path(row['workdir']) / 'configs' / 'daxm.yaml')
else:
    cfg = DW.config_for_scan(wirescans, index, detfile=DETFILE, calibfile=None, nb_cpus=NB_CPUS, **QUICKDAXM)
cfg['wirescalib']['fluo_element'] = FLUO_ELEMENT
cfg['wirescalib']['fluo_thickness'] = FLUO_THICKNESS
cfg['wirescalib']['fluo_layer'] = FLUO_LAYER
method = CRUDE_METHOD if CRUDE_METHOD != 'auto' else ('dips' if DW.is_reference_scan(row['dataset']) else 'envelope')
print('results folder:', cfg['output']['folder'])
print('crude calibration method:', method)

# STEP 3: dips of the Laue spots (quick daxm without wires calibration)

Blue: profile of a spot minus its local background, red/green: entry/exit edges of the dips. Orange: local background (fluorescence: its smooth dips are the data of the fine calibration, STEP 5).

In [ ]:
MODE

In [ ]:
if MODE == 'show':
    results = DW.load_quickdaxm(cfg)
else:
    results = DW.run_quickdaxm(cfg, resource='kernel', nb_cpus=NB_CPUS, redo=REDO)
print(f"{len(results['spots']['X'])} spots, {len(results['dips']['spot'])} dips")

SPOTS = list(range(6))
plt.close('all')
fig, axes = plt.subplots(len(SPOTS) // 2, 2, figsize=(14, 3.2 * (len(SPOTS) // 2)))
for ax, spot in zip(axes.ravel(), SPOTS):
    QD.plot_profile(results, spot, ax)
fig.tight_layout()
plt.show()

# STEP 4: crude calibration from the dips of the Laue spots

- `dips` (Ge): the dips of each wire lie on a band of the plot *image index of dip* vs *pixel Y*; least squares fit of `h`, `p0`, `f1` of each wire and `R` to the dips edges, all the sources at `ybeam = 0`
- `envelope` (any sample): for each band, the line beyond which there are no dips (only `ENVELOPE_OUTLIERS` of them): the shallowest grains of the region of the detector of the wire are at `ybeam = 0`. Right plot: vertical distance of the dips to the line of their wire (images, about microns along the beam for a step of 1 µm): sharp start at 0 expected

Wire 0: largest pixel Y of its shadow (closest to the sample surface). A wire seen during a part of the scan only has few dips: check it, exclude it (`EXCLUDED_WIRES`) if needed.

In [ ]:
plt.close('all')
if MODE != 'show':
    crude = DW.crude_wires(cfg, results, method, nb_wires=NB_WIRES, radius_each_wire=RADIUS_EACH_WIRE,
                           outliers=ENVELOPE_OUTLIERS, excluded_wires=EXCLUDED_WIRES)
    crude_calib = crude['calibfile']
    if method == 'envelope':
        QD.plot_envelope(results, crude)
else:
    crude_calib = str(Path(DW.calib_files(cfg)['folder']) / ('wires_from_dips.calib' if method == 'dips' else 'wires_from_envelope.calib'))
plt.show()

In [ ]:
# dips colored by their position along the beam with the crude wires, and wires lines for ybeam 0 and 50 µm
plt.close('all')
check = dict(results, calibfile=crude_calib)
check['dips'], check['spots'] = QD.analyse(results['signal'], results['xy'], QD.geometry_of_results(check), results['params'])
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
QD.plot_abacus(check, axes[0], ybeams=(0, 50))
QD.plot_histogram(check, axes[1], step=2.)
fig.tight_layout()
plt.show()

In [ ]:
IMAGES = None   # image indices of the Laue patterns shown with the predicted shadows (None: 3 images of the scan)

plt.close('all')
geometry = QD.WireGeometry.from_files(crude_calib, cfg['detector']['detfile'], results['wire_positions'])
nbimages = len(results['wire_positions'])
images = IMAGES or [nbimages // 6, nbimages // 2, 5 * nbimages // 6]
fig, axes = plt.subplots(1, len(images), figsize=(6 * len(images), 6))
for ax, imageindex in zip(np.atleast_1d(axes), images):
    QD.plot_image_shadows(cfg['scan'], geometry, imageindex, ax=ax)
fig.tight_layout()
plt.show()

# STEP 5: fine calibration with the fluorescence (option)

**Check of the profiles** (`CHECK_FLUO_PROFILES`): the fluorescence intensity profiles that the refinement will fit, at the points of a 3x3 grid per wire, in the windows of wire positions predicted by the crude wires. Each profile is fitted by the template of a fluorescence dip (sharp edge on the side of the surface, exponential tail of the deeper sources, shadow width predicted by the wires at this pixel). Usable (green): no Laue spot near the box, deep enough dip, above the noise, complete in its window, small residual of the template (it looks like a fluorescence dip), no spike, sharp edge close to the predicted one, shadow width close to the predicted one. With `REPLACE_BAD_POINTS`, an unusable point is replaced by the best usable point probed nearby, along X first at the same Y (as the background pixels of the PixelMonitoring notebooks), then towards the middle of the shadow region of the wire; the refinement uses the same replacements. Points still red are listed with the reason. A layered sample may show 2 dips (e.g. Cr coating and Zr below), unless one fluorescence is cut by the energy threshold of the detector: the model has one fluorescent element (`FLUO_ELEMENT`).

**Refinement** (`REFINE_FLUO`): `FLUO_STAGES` stages of `calib.py` (2x2, 3x3, 6x6 points per wire), starting from the crude wires. Fast optimizer (`LaueTools/Daxm/fastcalib.py`): same model and results, the amplitudes of the profiles solved exactly and the jacobian computed in parallel; `FLUO_RESOURCE = 'magnifix'` runs it as a SLURM job on a node of 192 cpus. The refined .calib file is the reference for the other scans of the sample: notebook 2 (`LOCAL_FLUO_REFINE`) or `DW.refine_wires_local()` refine only p0 (or h, p0) at another place, in seconds. With `FLUO_LAYER` (coated sample, e.g. Cr on Zr), the model has a uniform layer above the fluorescent material, whose thickness (common) and fraction of the signal (per profile) are fitted with the wires (`LaueTools/Daxm/fluolayer.py`); they are written in the .calib files (key `fluo_layer`). Without it, the signal of the layer pulls the depth zero towards the surface (~1.3 µm for Zr5) and blurs the edges of the dips. `FLUO_WIRES = [0, 1, 2]`: only these wires refined, the others keep their crude parameters and are excluded from the depth analysis and the reconstruction.

In [ ]:
plt.close('all')
if CHECK_FLUO_PROFILES and MODE != 'show':
    profiles_table = DW.check_fluo_profiles(cfg, crude_calib, dims=(3, 3), replace_bad_points=REPLACE_BAD_POINTS,
                                            probe_all=PROBE_ALL_POINTS)
    display(profiles_table)
plt.show()

In [ ]:
REFINE_FLUO

In [ ]:
REFINE_FLUO = True


fine_calib = None
if REFINE_FLUO and MODE != 'show':
    stages = DW.refine_wires_fluo_job(cfg, resource=FLUO_RESOURCE, nb_cpus=NB_CPUS, init_calib=crude_calib,
                                      stages=DW.FLUO_STAGES[:FLUO_STAGES], wires=FLUO_WIRES,
                                      excluded_wires=EXCLUDED_WIRES, replace_bad_points=REPLACE_BAD_POINTS,
                                      probe_all=PROBE_ALL_POINTS)
    for stage in stages:
        print(f"stage {stage['stage']} {stage['dims']}: GoF {stage['gof']:.2f}, fitted depth dm {stage['dm']}, "
              f"{stage['seconds']:.0f} s -> {stage['calibfile']}")
    fine_calib = stages[-1]['calibfile']
else:
    previous = sorted(Path(DW.calib_files(cfg)['folder']).glob('*_stage*.calib'), key=lambda p: p.stat().st_mtime)
    fine_calib = str(previous[-1]) if previous else None
print('fine calibration:', fine_calib)

**Depth resolution** (`FLUO_RESOLUTION`): each fluorescence profile fitted with the refined wires fixed, by the source model blurred by a gaussian: blur sigma (instrument: beam, wire edge, pixel) and shift (local error of the calibration), in microns along the beam. For a single element reference (Ge) sigma is the instrument resolution; for a layered sample (Cr on Zr) the single source model is incomplete and sigma is an upper bound.

In [ ]:
plt.close('all')
if FLUO_RESOLUTION and fine_calib and MODE != 'show':
    resolution_table = DW.fluo_resolution(cfg, fine_calib, dims=(3, 3), replace_bad_points=REPLACE_BAD_POINTS)
    display(resolution_table.round(2))
plt.show()

# STEP 6: comparison of the calibrations

Wires parameters and positions along the beam of the Laue spots of this scan with each calibration (same dips). The shift of the median `ybeam` between two calibrations is the difference of their depth zeros, e.g. `envelope` (shallowest grains: top of a coating) vs fluorescence (top of the fluorescent element). The `2-wire spread` (spots masked by 2 wires) and the `mismatch` (entry vs exit edges) check the consistency of a calibration.

In [ ]:
plt.close('all')
calibrations = {f'crude ({method})': crude_calib}
if fine_calib:
    calibrations['fluorescence'] = fine_calib
comparison = DW.compare_calibrations(results, calibrations)
display(comparison.round(4))
plt.show()

# RESULT

The wire scans of the same sample folder use the calibration of this scan automatically (`CALIBFILE = 'auto'` in `2_quick_daxm` and `3_quick_daxm_grid`): the refined one if any, else the crude one. Or set `CALIBFILE` to one of the files printed below.

In [ ]:
DW.refresh_status(wirescans)
print('crude calibration      :', crude_calib)
print('fluorescence calibration:', fine_calib)
print('used by CALIBFILE = auto:', DW.existing_calibration(row['workdir']))